# Tarea 3: Heart Failure Prediction: Validación, Hiperparámetros y Comparación de Modelos

MDS7104: Aprendizaje de Maquinas - Otoño 2026

---

### Cuerpo Docente:

- Profesor: Francisco Vásquez L.
- Auxiliares: Álvaro Márquez y Diego Olguín Wende
- Ayudantes: Javiera Yañez y Tamara Carrasco


### Estudiante

- Felipe Muñoz M.

In [ ]:
!uv add numpy requests pandas matplotlib scipy scikit-learn ruff pre-commit

Resolved 63 packages in 20ms
Audited 58 packages in 16ms


In [3]:
!uv add ipykernel ipython matplotlib-inline

Resolved 130 packages in 24ms
Audited 127 packages in 26ms


In [ ]:
import os

import matplotlib  # noqa: F401
import numpy as np  # noqa: F401
import pandas as pd  # noqa: F401

os.environ.pop("MPLBACKEND", None)


matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: F401
from scipy.optimize import minimize  # noqa: F401
from scipy.stats import beta as beta_dist  # noqa: F401
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # noqa: F401

In [ ]:
from sklearn.model_selection import train_test_split  # noqa: F401

In [ ]:
import plotly.express as px  # noqa: F401
import plotly.graph_objects as go  # noqa: F401
import plotly.io as pio  # noqa: F401
from plotly.subplots import make_subplots  # noqa: F401

# pio.renderers.default = "png"

In [37]:
# %matplotlib inline

# b) Particionamiento y exploración

Se procede a importar la base de datos

In [6]:
# Se abre el Dataframe
df = pd.read_csv("/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/tareas/tarea3/heart.csv")
df.head(5)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


Se realiza la partición de los datos, donde el 80% va al Train, y el 20% va al Test usando stratify

In [7]:
# Se separan las características (X) y la variable objetivo (y = HeartDisease)
features = [col for col in df.columns if col != "HeartDisease"]

# Se fijan las variables predictoras (X) y la variable objetivo (y)
X = df[features].values
y = df["HeartDisease"].values

# Se realiza el Split entre Train y Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=7,  # Se fija la semilla para reproducibilidad
    stratify=y,
)

Se verifican la proporción de clases y estadísticas descriptivas de HeartDiseare, y la Media y Desviación estandar de Cholesterol y Age. 

In [8]:
# Proporción de clases en train y test de HeartDisease
print("Proporción de clases en Train:")
print(pd.Series(y_train).value_counts(normalize=True))
print("\nProporción de clases en Test:")
print(pd.Series(y_test).value_counts(normalize=True))

Proporción de clases en Train:
1    0.553134
0    0.446866
Name: proportion, dtype: float64

Proporción de clases en Test:
1    0.554348
0    0.445652
Name: proportion, dtype: float64


In [9]:
X_train

array([[39, 'F', 'NAP', ..., 'N', 0.0, 'Flat'],
       [51, 'M', 'ASY', ..., 'Y', 1.2, 'Flat'],
       [64, 'M', 'ASY', ..., 'Y', 0.5, 'Flat'],
       ...,
       [45, 'M', 'TA', ..., 'N', 1.2, 'Flat'],
       [60, 'M', 'ASY', ..., 'Y', 1.5, 'Down'],
       [57, 'M', 'ASY', ..., 'Y', 3.0, 'Flat']],
      shape=(734, 11), dtype=object)

In [10]:
# Se calcula la Media y Desviación Estándar de cada característica en el conjunto de entrenamiento y Test
posición_característica = [0, 4]

print("Conjunto de Entrenamiento:")
for i in posición_característica:
    print(f"Característica {features[i]} - Media: {np.mean(X_train[:, i]):.2f}, Desviación Estándar: {np.std(X_train[:, i]):.2f}")

print("\nConjunto de Test:")
for i in posición_característica:
    print(f"Característica {features[i]} - Media: {np.mean(X_test[:, i]):.2f}, Desviación Estándar: {np.std(X_test[:, i]):.2f}")

Conjunto de Entrenamiento:
Característica Age - Media: 53.51, Desviación Estándar: 9.27
Característica Cholesterol - Media: 199.33, Desviación Estándar: 108.75

Conjunto de Test:
Característica Age - Media: 53.53, Desviación Estándar: 10.02
Característica Cholesterol - Media: 196.68, Desviación Estándar: 111.54


In [16]:
# Se realiza un histograma de la caracteristica "Cholesterol" de Train y Test
fig = make_subplots(rows=2, cols=1, subplot_titles=("Histograma de Cholesterol en Train", "Histograma de Cholesterol en Test"))

fig.add_trace(go.Histogram(x=X_train[:, 4], name="Train"), row=1, col=1)
fig.add_trace(go.Histogram(x=X_test[:, 4], name="Test", marker_color="red"), row=2, col=1)
fig.update_yaxes(title_text="Frecuencia", row=1, col=1)
fig.update_yaxes(title_text="Frecuencia", row=2, col=1)

fig.update_layout(height=600, showlegend=False)
fig.show()